# Strength Scan WeightWatcher

## Configuration

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

SCAN_ROOT = Path(
    os.environ.get(
        "WWGPT_STRENGTH_SCAN_ROOT",
        "/tmp/wwpgd_strength_scan",
    )
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wwgpt.strength_scan_analysis import resolve_scan_root, analyze_strength_scan
scan_root = resolve_scan_root(SCAN_ROOT)
analysis_dir = analyze_strength_scan(scan_root)
fig_dir = scan_root / "analysis" / "notebook_weightwatcher"
fig_dir.mkdir(parents=True, exist_ok=True)
print(scan_root)

## Immediate spectral-record audit

## Fit-quality audit

## Alpha before and after projection

## Distance to target alpha

## Per-layer response

## Alpha trajectories by scan with seed Bollinger Bands

For every WW-PGD scan strength, collect WeightWatcher `alpha` values from each run's `spectral.csv`, then plot:

- the per-seed average layer alpha over training step with a mean ±2 seed-standard-deviation Bollinger band; and
- each individual layer alpha trajectory over training step with its own mean ±2 seed-standard-deviation Bollinger band across seeds.

In [ ]:
import re
import numpy as np

runs = pd.read_csv(analysis_dir / "strength_scan_runs.csv")


def _safe_strength_label(strength):
    if pd.isna(strength):
        return "unknown"
    return re.sub(r"[^0-9A-Za-z_.-]+", "_", f"{float(strength):g}")


def _load_wwpgd_spectral_trajectories(runs_df):
    frames = []
    for _, run_row in runs_df[runs_df.optimizer_family == "wwpgd"].iterrows():
        spectral_path = Path(run_row.run_dir) / "spectral.csv"
        if not spectral_path.exists():
            continue
        spectral = pd.read_csv(spectral_path)
        required = {"step", "alpha"}
        if not required.issubset(spectral.columns):
            continue
        layer_col = "longname" if "longname" in spectral.columns else "name" if "name" in spectral.columns else None
        if layer_col is None:
            continue
        keep = ["step", layer_col, "alpha"]
        if "tokens_seen" in spectral.columns:
            keep.append("tokens_seen")
        if "valid_for_science" in spectral.columns:
            keep.append("valid_for_science")
        frame = spectral[keep].copy()
        frame = frame.rename(columns={layer_col: "layer_name"})
        frame["alpha"] = pd.to_numeric(frame["alpha"], errors="coerce")
        frame["step"] = pd.to_numeric(frame["step"], errors="coerce")
        if "tokens_seen" in frame.columns:
            frame["tokens_seen"] = pd.to_numeric(frame["tokens_seen"], errors="coerce")
        else:
            frame["tokens_seen"] = np.nan
        if "valid_for_science" in frame.columns:
            valid = frame["valid_for_science"].fillna(True).astype(bool)
            frame = frame[valid]
        frame = frame.dropna(subset=["step", "alpha", "layer_name"])
        if frame.empty:
            continue
        frame["seed"] = run_row.seed
        frame["strength"] = run_row.strength
        frame["scan_id"] = run_row.scan_id
        frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=["scan_id", "strength", "seed", "step", "tokens_seen", "layer_name", "alpha"])
    return pd.concat(frames, ignore_index=True)


alpha_trajectories = _load_wwpgd_spectral_trajectories(runs)
alpha_trajectories.to_csv(fig_dir / "wwpgd_layer_alpha_trajectories.csv", index=False)
display(alpha_trajectories.head())


def _bollinger_summary(df, group_cols, value_col="alpha"):
    out = (
        df.groupby(group_cols, dropna=False)[value_col]
        .agg(mean="mean", std="std", seed_count="count")
        .reset_index()
        .sort_values(group_cols)
    )
    out["std"] = out["std"].fillna(0.0)
    out["bollinger_lower"] = out["mean"] - 2.0 * out["std"]
    out["bollinger_upper"] = out["mean"] + 2.0 * out["std"]
    return out


if alpha_trajectories.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.text(0.5, 0.5, "No per-step WeightWatcher spectral.csv alpha trajectories found.", ha="center", va="center")
    ax.axis("off")
    fig.savefig(fig_dir / "wwpgd_alpha_trajectories_missing.png", bbox_inches="tight", dpi=150)
    plt.close(fig)
else:
    # Average alpha per seed/step first, then Bollinger-band those seed means within each scan.
    average_seed_step = (
        alpha_trajectories.groupby(["scan_id", "strength", "seed", "step"], dropna=False)
        .agg(average_alpha=("alpha", "mean"), tokens_seen=("tokens_seen", "max"), layer_count=("layer_name", "nunique"))
        .reset_index()
    )
    average_bands = _bollinger_summary(
        average_seed_step,
        ["scan_id", "strength", "step"],
        value_col="average_alpha",
    )
    average_bands.to_csv(fig_dir / "wwpgd_average_alpha_bollinger_bands_by_scan.csv", index=False)

    layer_seed_step = (
        alpha_trajectories.groupby(["scan_id", "strength", "layer_name", "seed", "step"], dropna=False)
        .agg(alpha=("alpha", "mean"), tokens_seen=("tokens_seen", "max"))
        .reset_index()
    )
    layer_bands = _bollinger_summary(
        layer_seed_step,
        ["scan_id", "strength", "layer_name", "step"],
        value_col="alpha",
    )
    layer_bands.to_csv(fig_dir / "wwpgd_layer_alpha_bollinger_bands_by_scan.csv", index=False)

    for strength, scan_avg in average_bands.groupby("strength", dropna=False):
        label = _safe_strength_label(strength)
        scan_avg = scan_avg.sort_values("step")
        x = scan_avg["step"].to_numpy(dtype=float)
        mean = scan_avg["mean"].to_numpy(dtype=float)
        lower = scan_avg["bollinger_lower"].to_numpy(dtype=float)
        upper = scan_avg["bollinger_upper"].to_numpy(dtype=float)
        fig, ax = plt.subplots(figsize=(10, 5.5))
        ax.plot(x, mean, marker="o", linewidth=2.2, label="mean of per-seed average layer alpha")
        ax.fill_between(x, lower, upper, alpha=0.18, label="Bollinger band: mean ± 2 seed std")
        ax.axhline(2.0, color="black", linestyle="--", linewidth=1, alpha=0.55, label="target alpha = 2")
        ax.set_title(f"WW-PGD strength={float(strength):g}: average layer alpha by training step")
        ax.set_xlabel("training step")
        ax.set_ylabel("WeightWatcher alpha")
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
        fig.savefig(fig_dir / f"wwpgd_strength_{label}_average_alpha_bollinger.png", bbox_inches="tight", dpi=150)
        plt.close(fig)

    for strength, scan_layers in layer_bands.groupby("strength", dropna=False):
        label = _safe_strength_label(strength)
        fig, ax = plt.subplots(figsize=(12, max(6, 0.22 * scan_layers["layer_name"].nunique())))
        cmap = plt.get_cmap("tab20", max(1, scan_layers["layer_name"].nunique()))
        for idx, (layer_name, layer_df) in enumerate(scan_layers.groupby("layer_name")):
            layer_df = layer_df.sort_values("step")
            x = layer_df["step"].to_numpy(dtype=float)
            mean = layer_df["mean"].to_numpy(dtype=float)
            lower = layer_df["bollinger_lower"].to_numpy(dtype=float)
            upper = layer_df["bollinger_upper"].to_numpy(dtype=float)
            color = cmap(idx)
            ax.plot(x, mean, linewidth=1.4, alpha=0.9, color=color, label=layer_name)
            ax.fill_between(x, lower, upper, color=color, alpha=0.08)
        ax.axhline(2.0, color="black", linestyle="--", linewidth=1, alpha=0.55, label="target alpha = 2")
        ax.set_title(f"WW-PGD strength={float(strength):g}: layer alphas by training step (mean ± 2 seed std)")
        ax.set_xlabel("training step")
        ax.set_ylabel("WeightWatcher alpha")
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=6, ncol=2, loc="best")
        fig.savefig(fig_dir / f"wwpgd_strength_{label}_layer_alpha_bollinger.png", bbox_inches="tight", dpi=150)
        plt.close(fig)

    display(average_bands)
    display(layer_bands.head())


## Projection-event response

## Projection magnitude versus spectral correction

## TraceLog preservation

## Strength-response interpretation

In [ ]:
spec=pd.read_csv(analysis_dir/"strength_scan_spectral.csv"); runs=pd.read_csv(analysis_dir/"strength_scan_runs.csv")
raw=[]
for d in runs[runs.optimizer_family=="wwpgd"].run_dir:
    p=Path(d)/"wwpgd_projection_spectral.csv"
    if p.exists(): raw.append(pd.read_csv(p))
raw=pd.concat(raw, ignore_index=True) if raw else pd.DataFrame()
display(raw); display(raw[raw[["alpha_before","alpha_after"]].isna().any(axis=1)] if len(raw) else raw); display(spec)
if len(raw):
    ax=raw.plot.scatter(x="alpha_before", y="alpha_after", c="scan_strength", colormap="viridis"); ax.figure.savefig(fig_dir/"alpha_before_after.png"); plt.close(ax.figure)
    for y,name in [("abs_alpha_error_change","alpha_error_change"),("relative_frobenius_change","fro_vs_alpha")]:
        ax=raw.plot.scatter(x="scan_strength" if y=="abs_alpha_error_change" else "relative_frobenius_change", y=y); ax.figure.savefig(fig_dir/f"{name}.png"); plt.close(ax.figure)
    for y,name in [("TraceLog_change","tracelog_change"),("D_before","fit_D_before"),("num_evals_before","eligible_fits"),("effective_blend_eta","etas")]:
        ax=raw.groupby("scan_strength")[y].median().plot(marker="o"); ax.figure.savefig(fig_dir/f"{name}.png"); plt.close(ax.figure)
if len(spec):
    for y,name in [("median_abs_alpha_error_change","median_error_change"),("fraction_layers_closer_to_target","fraction_closer"),("median_alpha_before","median_alpha_event")]:
        ax=spec.groupby("strength")[y].mean().plot(marker="o"); ax.figure.savefig(fig_dir/f"{name}.png"); plt.close(ax.figure)
print("Fit filtering is explicit; invalid fits remain in audit tables. Layers are not treated as independent seeds.")